In [2]:
import os
import subprocess
from pathlib import Path
from dotenv import load_dotenv
import anthropic

# Load environment variables
load_dotenv()

# Test Anthropic connection
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Paths — notebook is in notebooks/, so parent is the project root
CODEBASE_PATH = str(Path.cwd().parent / "mcp-gateway-registry")

message = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)

print(message.content[0].text)
print(f"✅ Anthropic client ready")
print(f"✅ Codebase path: {CODEBASE_PATH}")

Hello! I'm happy to meet you.
✅ Anthropic client ready
✅ Codebase path: /Users/animeliksetyan/Documents/DSAN_AIAgent/spring-2026-a03-Animeliksetyan/mcp-gateway-registry


In [14]:
QUESTION_TYPES = {
    "dependencies": ["dependencies", "requirements", "libraries", "packages", "pyproject", 
                     "imports", "modules", "installed", "pip", "toml", "third party"],
    "entrypoint": ["entry point", "main entry", "start file", "main file", "starts", 
                   "launch", "initialize", "main.py", "app.py", "where does it start", "kicks off"],
    "filetypes": ["programming languages", "file types", "languages used", "what languages", 
                  "tech stack", "extensions", "typescript", "yaml", "codebase uses"],
    "auth_flow": ["authentication flow", "token validation", "authorization", 
                  "how does auth", "login", "oauth", "access control", "permissions", 
                  "secure", "credentials"],
    "api_endpoints": ["api endpoints", "endpoints", "routes", "scopes", "api routes", 
                      "url paths", "request handlers", "controllers"],
    "architecture": ["add support", "how would you", "what files would", "modify", "implement", 
                     "integrate", "extend", "how to add", "what changes", "need to change", "design"]
}

def classify_query_keywords(question: str) -> str:
    """Classify using keyword matching with architecture priority check."""
    question_lower = question.lower()
    
    architecture_actions = ["add support", "how would you", "what files would", 
                           "modify", "implement", "integrate", "extend", 
                           "how to add", "what changes", "need to change", 
                           "which files", "design"]
    
    for keyword in architecture_actions:
        if keyword in question_lower:
            return "architecture"
    
    for qtype, keywords in QUESTION_TYPES.items():
        if qtype == "architecture":
            continue
        for keyword in keywords:
            if keyword in question_lower:
                return qtype
    
    return None

def classify_query_llm(question: str) -> str:
    """Fall back to LLM classification if keywords fail."""
    prompt = f"""You are a query classifier for a code Q&A system.
    
Classify the following question into exactly ONE of these categories:
- dependencies: questions about libraries, packages, requirements
- entrypoint: questions about main file, startup, bootstrap
- filetypes: questions about languages, tech stack, file extensions
- auth_flow: questions about authentication, authorization, tokens, JWT, OAuth, login
- api_endpoints: questions about routes, endpoints, HTTP methods, scopes
- architecture: questions about how to modify, extend, add features, design

Question: {question}

Reply with ONLY the category name, nothing else."""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=20,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip().lower()

def classify_query(question: str) -> str:
    """Hybrid classifier — keywords first, LLM fallback."""
    
    result = classify_query_keywords(question)
    
    if result:
        #print(f"  [Classifier] Keyword match → {result}")
        return result
    
    #print(f"  [Classifier] No keyword match, using LLM...")
    result = classify_query_llm(question)
    #print(f"  [Classifier] LLM classified → {result}")
    return result

# Test it on all 6 questions
test_questions = [
    "What Python dependencies does this project use?",
    "What is the main entry point file for the registry service?",
    "What programming languages and file types are used in this repository?",
    "How does the authentication flow work, from token validation to user authorization?",
    "What are all the API endpoints available in the registry service and what scopes do they require?",
    "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?"
]

for q in test_questions:
    qtype = classify_query(q)
    print(f"Q: {q[:60]}...")
    print(f"Type: {qtype}\n")

Q: What Python dependencies does this project use?...
Type: dependencies

Q: What is the main entry point file for the registry service?...
Type: entrypoint

Q: What programming languages and file types are used in this r...
Type: filetypes

Q: How does the authentication flow work, from token validation...
Type: auth_flow

Q: What are all the API endpoints available in the registry ser...
Type: api_endpoints

Q: How would you add support for a new OAuth provider (e.g., Ok...
Type: architecture



In [15]:
def run_bash_command(command: list) -> str:
    """Run a bash command and return output as string."""
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            cwd=os.getcwd()
        )
        if result.returncode != 0:
            return f"Command error: {result.stderr[:500]}"
        return result.stdout
    except Exception as e:
        return f"Exception: {str(e)}"

def truncate_output(output: str, max_chars: int = 3000) -> str:
    """Truncate large outputs to avoid token limit."""
    if len(output) <= max_chars:
        return output
    return output[:max_chars] + f"\n... [truncated, {len(output)} total chars]"

def get_context_for_query(question_type: str) -> str:
    """Map question type to bash commands and execute them."""
    
    context_parts = []
    
    if question_type == "dependencies":
        #print("  [Executor] Running: cat pyproject.toml")
        output = run_bash_command(["cat", f"{CODEBASE_PATH}/pyproject.toml"])
        context_parts.append(f"=== DEPENDENCIES (pyproject.toml) ===\n{output}")
        output2 = run_bash_command(["find", f"{CODEBASE_PATH}", "-name", "package.json", "-not", "-path", "*/node_modules/*"])
        context_parts.append(f"=== JS/TS PACKAGE FILES ===\n{output2}")
        output3 = run_bash_command(["cat", f"{CODEBASE_PATH}/cli/package.json"])
        context_parts.append(f"=== CLI DEPENDENCIES (package.json) ===\n{truncate_output(output3, 1000)}")
    
    elif question_type == "entrypoint":
        #print("  [Executor] Running: grep FastAPI(")
        output = run_bash_command([
            "grep", "-r", "FastAPI(", f"{CODEBASE_PATH}/registry/",
            "--include=*.py", "-l"
        ])
        context_parts.append(f"=== FILES CONTAINING FastAPI() ===\n{output}")
        
        #print("  [Executor] Running: grep __main__")
        output2 = run_bash_command([
            "grep", "-r", "__main__", f"{CODEBASE_PATH}/registry/",
            "--include=*.py", "-l"
        ])
        context_parts.append(f"=== FILES WITH __main__ ===\n{output2}")
    
    elif question_type == "filetypes":
        #print("  [Executor] Running: find all files")
        output = run_bash_command([
            "find", f"{CODEBASE_PATH}", "-type", "f",
            "-not", "-path", "*/.git/*"
        ])
        lines = output.strip().split("\n")
        extensions = {}
        for line in lines:
            if "." in line:
                ext = line.split(".")[-1].lower()
                extensions[ext] = extensions.get(ext, 0) + 1
        ext_summary = "\n".join([f".{k}: {v} files" 
                                  for k, v in sorted(extensions.items(), 
                                  key=lambda x: x[1], reverse=True)])
        context_parts.append(f"=== FILE TYPES IN REPOSITORY ===\n{ext_summary}")
    
    elif question_type == "auth_flow":
        #print("  [Executor] Running: grep token in auth_server/")
        output1 = run_bash_command([
            "grep", "-r", "token", f"{CODEBASE_PATH}/auth_server/",
            "--include=*.py", "-n"
        ])
        context_parts.append(f"=== TOKEN HANDLING (auth_server) ===\n{truncate_output(output1, 1000)}")
        
        #print("  [Executor] Running: grep token in registry/auth/")
        output2 = run_bash_command([
            "grep", "-r", "token", f"{CODEBASE_PATH}/registry/auth/",
            "--include=*.py", "-n"
        ])
        context_parts.append(f"=== TOKEN HANDLING (registry/auth) ===\n{truncate_output(output2, 1000)}")
        
        #print("  [Executor] Running: grep token in registry/middleware/")
        output3 = run_bash_command([
            "grep", "-r", "token", f"{CODEBASE_PATH}/registry/middleware/",
            "--include=*.py", "-n"
        ])
        context_parts.append(f"=== MIDDLEWARE AUTH ===\n{truncate_output(output3, 1000)}")

        #print("  [Executor] Running: grep auth in docs/")
        output4 = run_bash_command([
            "grep", "-r", "auth\|token\|oauth",
            f"{CODEBASE_PATH}/docs/",
            "-l"
        ])
        context_parts.append(f"=== AUTH DOCUMENTATION FILES ===\n{output4}")
    
    elif question_type == "api_endpoints":
        #print("  [Executor] Running: list all route files")
        route_files = run_bash_command([
            "find", f"{CODEBASE_PATH}/registry/api/",
            "-name", "*.py", "-type", "f"
        ])
        context_parts.append(f"=== ROUTE FILES ===\n{route_files}")
        
        #print("  [Executor] Running: grep route decorators in each file")
        all_routes = []
        for route_file in route_files.strip().split("\n"):
            if route_file.endswith(".py") and "__init__" not in route_file:
                output = run_bash_command([
                    "grep", "-n", "-A", "1",
                    "@router.get\\|@router.post\\|@router.put\\|@router.delete\\|@router.patch",
                    route_file
                ])
                if output.strip():
                    all_routes.append(f"--- {route_file.split('/')[-1]} ---\n{truncate_output(output, 500)}")
        
        context_parts.append(f"=== ALL ROUTES BY FILE ===\n" + "\n".join(all_routes))
        
        #print("  [Executor] Running: grep auth dependencies on routes")
        output2 = run_bash_command([
            "grep", "-r", "Depends", f"{CODEBASE_PATH}/registry/api/",
            "--include=*.py", "-n"
        ])
        context_parts.append(f"=== AUTH DEPENDENCIES ON ROUTES ===\n{truncate_output(output2, 2000)}")
        
        #print("  [Executor] Running: grep scope system in auth dependencies")
        output3 = run_bash_command([
            "grep", "-n", "scope\\|permission",
            f"{CODEBASE_PATH}/registry/auth/dependencies.py"
        ])
        context_parts.append(f"=== SCOPE SYSTEM ===\n{truncate_output(output3, 1000)}")
    
    elif question_type == "architecture":
        #print("  [Executor] Running: find credentials-provider/")
        output1 = run_bash_command([
            "find", f"{CODEBASE_PATH}/credentials-provider/",
            "-type", "f", "-name", "*.py"
        ])
        context_parts.append(f"=== CREDENTIALS PROVIDER FILES ===\n{output1}")
        
        #print("  [Executor] Running: find auth_server/providers/")
        output2 = run_bash_command([
            "find", f"{CODEBASE_PATH}/auth_server/providers/",
            "-type", "f"
        ])
        context_parts.append(f"=== AUTH SERVER PROVIDER FILES ===\n{output2}")
        
        #print("  [Executor] Running: grep provider in auth_server/")
        output3 = run_bash_command([
            "grep", "-r", "provider", f"{CODEBASE_PATH}/auth_server/",
            "--include=*.py", "-n"
        ])
        context_parts.append(f"=== PROVIDER REFERENCES ===\n{truncate_output(output3, 1000)}")
    
    else:
        #print("  [Executor] Running: general search")
        output = run_bash_command([
            "grep", "-r", question_type, f"{CODEBASE_PATH}/",
            "--include=*.py", "-l"
        ])
        context_parts.append(f"=== GENERAL SEARCH ===\n{output}")
    
    return "\n\n".join(context_parts)

# Test it on all 6 question types
print("=== Testing Bash Executor ===\n")
for q in test_questions:
    qtype = classify_query(q)
    print(f"\nQ: {q[:60]}")
    print(f"Type: {qtype}")
    print("-" * 50)

=== Testing Bash Executor ===


Q: What Python dependencies does this project use?
Type: dependencies
--------------------------------------------------

Q: What is the main entry point file for the registry service?
Type: entrypoint
--------------------------------------------------

Q: What programming languages and file types are used in this r
Type: filetypes
--------------------------------------------------

Q: How does the authentication flow work, from token validation
Type: auth_flow
--------------------------------------------------

Q: What are all the API endpoints available in the registry ser
Type: api_endpoints
--------------------------------------------------

Q: How would you add support for a new OAuth provider (e.g., Ok
Type: architecture
--------------------------------------------------


In [6]:
# Debug - see how many routes per file
route_files_list = route_files.strip().split("\n")

for route_file in route_files_list:
    if route_file.endswith(".py") and "__init__" not in route_file:
        output = run_bash_command(["grep", "-n", "@router", route_file])
        line_count = len(output.strip().split("\n")) if output.strip() else 0
        print(f"{route_file.split('/')[-1]}: {line_count} routes")

federation_routes.py: 10 routes
federation_export_routes.py: 4 routes
virtual_server_routes.py: 10 routes
search_routes.py: 1 routes
config_routes.py: 3 routes
management_routes.py: 10 routes
registry_routes.py: 3 routes
wellknown_routes.py: 1 routes
internal_routes.py: 5 routes
server_routes.py: 40 routes
agent_routes.py: 13 routes
skill_routes.py: 16 routes
peer_management_routes.py: 14 routes


In [7]:
# Debug - see just the route paths per file
route_files_list = route_files.strip().split("\n")

for route_file in route_files_list:
    if route_file.endswith(".py") and "__init__" not in route_file:
        # Get just the route decorator lines
        output = run_bash_command(["grep", "-n", '\"/', route_file])
        if output.strip():
            print(f"\n--- {route_file.split('/')[-1]} ---")
            print(output[:500])


--- federation_routes.py ---
34:@router.get("/federation/config", tags=["federation"], summary="Get federation configuration")
72:    "/federation/config",
145:    "/federation/config/{config_id}",
192:    "/federation/config/{config_id}", tags=["federation"], summary="Delete federation configuration"
231:    "/federation/configs", tags=["federation"], summary="List all federation configurations"
255:    "/federation/config/{config_id}/anthropic/servers",
309:    "/federation/config/{config_id}/anthropic/servers/{server_nam

--- federation_export_routes.py ---
29:router = APIRouter(prefix="/api/federation", tags=["federation"])
359:@router.get("/health")
378:    "/servers",
476:        endpoint="/api/federation/servers",
491:    "/agents",
585:        endpoint="/api/federation/agents",
600:    "/security-scans",
716:        endpoint="/api/federation/security-scans",


--- virtual_server_routes.py ---
104:    if raw_path.startswith("/virtual/"):
107:        normalized = f"/{raw_path}"


In [8]:
# Debug - get only actual route decorator lines
for route_file in route_files_list:
    if route_file.endswith(".py") and "__init__" not in route_file:
        output = run_bash_command([
            "grep", "-n", 
            "@router.get\|@router.post\|@router.put\|@router.delete\|@router.patch",
            route_file
        ])
        if output.strip():
            print(f"\n--- {route_file.split('/')[-1]} ---")
            print(output)


--- federation_routes.py ---
34:@router.get("/federation/config", tags=["federation"], summary="Get federation configuration")
71:@router.post(
144:@router.put(
191:@router.delete(
230:@router.get(
254:@router.post(
308:@router.delete(
359:@router.post(
413:@router.delete(
464:@router.post("/federation/sync", tags=["federation"], summary="Trigger manual federation sync")


--- federation_export_routes.py ---
359:@router.get("/health")
377:@router.get(
490:@router.get(
599:@router.get(


--- virtual_server_routes.py ---
129:@router.get(
167:@router.get(
206:@router.post(
253:@router.get(
275:@router.get(
298:@router.post(
354:@router.put(
414:@router.delete(
455:@router.post(
513:@router.get(


--- search_routes.py ---
353:@router.post(


--- config_routes.py ---
351:@router.get(
431:@router.get(
661:@router.get(


--- management_routes.py ---
126:@router.get("/iam/users", response_model=UserListResponse)
159:@router.post("/iam/users/m2m")
181:@router.post("/iam/users/human")
214:@rout

In [18]:
def generate_answer(question: str, context: str, question_type: str) -> str:
    """Send question + context to Claude and get an answer."""
    
    prompt = f"""You are a code expert analyzing the mcp-gateway-registry codebase.

Use the following context retrieved from the codebase to answer the question.
Always cite specific file names and line numbers in your answer.
Be precise and technical.

Context:
{context}

Question: {question}

Instructions:
- Answer based ONLY on the provided context
- Cite specific files like: (found in registry/main.py, line 42)
- If context is insufficient, say what additional files would need to be checked
- Be concise but complete
"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=4000,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.content[0].text

# Test on all 6 questions
print("=== Generating Answers for All 6 Questions ===\n")

results = []

for q in test_questions:
    print(f"Q: {q}")
    print("Processing...")
    
    # Step 1: Classify
    qtype = classify_query(q)
    
    # Step 2: Get context
    context = get_context_for_query(qtype)
    
    # Step 3: Generate answer
    answer = generate_answer(q, context, qtype)
    
    print(f"Answer:\n{answer}")
    print("-" * 60)
    
    results.append({
        "question": q,
        "type": qtype,
        "answer": answer
    })

=== Generating Answers for All 6 Questions ===

Q: What Python dependencies does this project use?
Processing...
Answer:
# Python Dependencies

Based on the `pyproject.toml` file, this project uses the following Python dependencies:

## Core Framework & Web
- **fastapi** >=0.115.12 - Web framework
- **uvicorn[standard]** >=0.34.2 - ASGI server
- **httpx** >=0.27.0 - Async HTTP client
- **aiohttp** >=3.8.0 - Async HTTP library
- **websockets** >=15.0.1 - WebSocket support

## MCP & AI/LLM
- **mcp** >=1.9.3 - Model Context Protocol
- **langchain-mcp-adapters** >=0.0.11 - LangChain MCP integration
- **langgraph** >=0.4.3 - Graph-based agent framework
- **langchain-aws** >=0.2.23 - LangChain AWS integration
- **langchain-anthropic** >=0.3.17 - LangChain Anthropic integration
- **litellm** >=1.50.0 - LLM abstraction layer

## Machine Learning & Vector Search
- **sentence-transformers** >=3.0.0 - Sentence embeddings
- **faiss-cpu** >=1.7.4 - Vector search
- **scikit-learn** >=1.3.0 - ML util

In [20]:
# Save all results to part1_results.txt
output_path = str(Path.cwd().parent /"part1_results.txt")

with open(output_path, "w") as f:
    f.write("=" * 60 + "\n")
    f.write("PART 1: CODE Q&A SYSTEM RESULTS\n")
    f.write("=" * 60 + "\n\n")
    
    for i, result in enumerate(results, 1):
        f.write(f"QUESTION {i}: {result['question']}\n")
        f.write(f"TYPE: {result['type']}\n")
        f.write("-" * 40 + "\n")
        f.write(f"ANSWER:\n{result['answer']}\n")
        f.write("=" * 60 + "\n\n")

print(f"Results saved to {output_path}")

Results saved to /Users/animeliksetyan/Documents/DSAN_AIAgent/spring-2026-a03-Animeliksetyan/part1_results.txt
